## CRPBSFinder webcrawler code and promoter integration
This code is designed to break fasta files into smaller, uploadable sizes, process them with the CRPBSFinder web interface (https://awi.cuhk.edu.cn/~CRPBSFinder), and then merge the resulting outputs. These can then be integrated into a dataframe with promoter and operon region information to identify genes potentially regulated by CRP. 

The webcrawler was built with lots of assistance from Claude, and we recommend using AI fro any troubleshooting assistance. Additional requirements for this code include 'requests', 'beautifulsoup4', 'lxml', and 'selemium webdriver-manager':
#pip install requests beautifulsoup4 lxml
#pip install selenium webdriver-manager

The code assumes and existing file structure with directories "results", "merged_results", and "split_fasta_files" located in the current working directory. To resume and analysis using 'parallel_complete_workflow', any files located within these directories may be acted upon, so if you may wish to remove previously analyzed files if you know they were fully analyzed to avoid unnecessary repetition. 

In [6]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pathlib import Path
import time
import os
import csv
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================================
# PART 1: FASTA SPLITTING WITH CONTIG TRACKING
# ============================================================================

class AdvancedFastaSplitter:
    def __init__(self, max_size_kb=30, max_contig_size_kb=25):
        """
        Initialize the FASTA splitter with contig splitting capability
        
        Args:
            max_size_kb: Maximum size for each output file in kilobytes
            max_contig_size_kb: Maximum size for a single contig before splitting it
        """
        self.max_size_bytes = max_size_kb * 1024
        self.max_contig_size_bytes = max_contig_size_kb * 1024
        self.split_tracking = {}  # Track contig splits for position correction
    
    def read_fasta_contigs(self, file_path):
        """
        Read a FASTA file and return a list of contigs
        Each contig is a tuple of (header, sequence)
        """
        contigs = []
        current_header = None
        current_sequence = []
        
        with open(file_path, 'r') as f:
            for line in f:
                line = line.rstrip('\n\r')
                
                if line.startswith('>'):
                    if current_header is not None:
                        contigs.append((current_header, ''.join(current_sequence)))
                    current_header = line
                    current_sequence = []
                else:
                    current_sequence.append(line)
            
            if current_header is not None:
                contigs.append((current_header, ''.join(current_sequence)))
        
        return contigs
    
    def split_large_contig(self, header, sequence, chunk_size=None):
        """
        Split a large contig into smaller chunks
        
        Args:
            header: Contig header line (with >)
            sequence: Full sequence string
            chunk_size: Size of each chunk (defaults to max_contig_size_bytes)
            
        Returns:
            List of tuples: [(new_header, chunk_sequence, original_contig_name, offset), ...]
        """
        if chunk_size is None:
            chunk_size = self.max_contig_size_bytes
        
        # Extract contig name from header (remove '>')
        contig_name = header[1:].split()[0]
        
        chunks = []
        sequence_length = len(sequence)
        num_chunks = (sequence_length + chunk_size - 1) // chunk_size
        
        for i in range(num_chunks):
            start = i * chunk_size
            end = min((i + 1) * chunk_size, sequence_length)
            chunk_seq = sequence[start:end]
            
            # Create new header with chunk info
            chunk_header = f">{contig_name}_chunk{i+1:03d}_of_{num_chunks:03d}_offset{start}"
            
            chunks.append((chunk_header, chunk_seq, contig_name, start))
        
        return chunks
    
    def get_contig_size(self, contig):
        """Calculate the size of a contig in bytes when written to file"""
        header, sequence = contig
        return len(header) + 1 + len(sequence) + 1
    
    def process_contigs(self, contigs):
        """
        Process contigs: split large ones, keep small ones
        
        Returns:
            List of processed contigs with tracking info:
            [(header, sequence, original_name, offset), ...]
        """
        processed = []
        
        for header, sequence in contigs:
            contig_name = header[1:].split()[0]
            contig_size = len(sequence)
            
            if contig_size > self.max_contig_size_bytes:
                # Split this contig
                print(f"    Splitting large contig '{contig_name}' ({contig_size / 1024:.2f} KB)")
                chunks = self.split_large_contig(header, sequence)
                processed.extend(chunks)
                print(f"      -> Created {len(chunks)} chunk(s)")
            else:
                # Keep as is
                processed.append((header, sequence, contig_name, 0))
        
        return processed
    
    def split_contigs_into_chunks(self, processed_contigs):
        """
        Group processed contigs into file chunks that don't exceed max_size_bytes
        """
        chunks = []
        current_chunk = []
        current_size = 0
        
        for contig_data in processed_contigs:
            header, sequence = contig_data[0], contig_data[1]
            contig_size = len(header) + 1 + len(sequence) + 1
            
            if current_size + contig_size > self.max_size_bytes and current_chunk:
                chunks.append(current_chunk)
                current_chunk = [contig_data]
                current_size = contig_size
            else:
                current_chunk.append(contig_data)
                current_size += contig_size
        
        if current_chunk:
            chunks.append(current_chunk)
        
        return chunks
    
    def write_chunk(self, contig_data_list, output_path):
        """
        Write a chunk of contigs to a file and save tracking info
        """
        with open(output_path, 'w') as f:
            for header, sequence, original_name, offset in contig_data_list:
                f.write(f"{header}\n{sequence}\n")
        
        # Save tracking info
        tracking_path = str(output_path) + ".tracking"
        with open(tracking_path, 'w') as f:
            f.write("chunk_name\toriginal_contig\toffset\n")
            for header, sequence, original_name, offset in contig_data_list:
                chunk_name = header[1:].split()[0]  # Remove '>' and get just the name
                f.write(f"{chunk_name}\t{original_name}\t{offset}\n")
    
    def split_file(self, input_file, output_dir):
        """
        Split a single FASTA file into multiple files
        """
        input_path = Path(input_file)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\nProcessing: {input_path.name}")
        
        # Read all contigs
        contigs = self.read_fasta_contigs(input_file)
        print(f"  Found {len(contigs)} contig(s)")
        
        # Process contigs (split large ones)
        processed_contigs = self.process_contigs(contigs)
        
        # Group into file chunks
        chunks = self.split_contigs_into_chunks(processed_contigs)
        print(f"  Created {len(chunks)} output file(s)")
        
        # Write chunks to files
        output_files = []
        base_name = input_path.stem
        
        for i, chunk in enumerate(chunks, 1):
            if len(chunks) == 1:
                output_name = f"{base_name}.fasta"
            else:
                output_name = f"{base_name}_part{i:03d}.fasta"
            
            output_path = output_dir / output_name
            self.write_chunk(chunk, output_path)
            
            actual_size = os.path.getsize(output_path)
            num_contigs = len(chunk)
            
            print(f"    Part {i}: {num_contigs} sequence(s), {actual_size / 1024:.2f} KB -> {output_name}")
            output_files.append(str(output_path))
        
        return output_files
    
    def split_directory(self, input_dir, output_dir):
        """Split all FASTA files in a directory"""
        input_dir = Path(input_dir)
        output_dir = Path(output_dir)
        
        fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
        fasta_files = []
        for ext in fasta_extensions:
            fasta_files.extend(input_dir.glob(ext))
        
        fasta_files = sorted(fasta_files)
        
        if not fasta_files:
            print(f"No FASTA files found in {input_dir}")
            return {}
        
        print("="*60)
        print(f"Found {len(fasta_files)} FASTA file(s) to process")
        print(f"Maximum file size: {self.max_size_bytes / 1024} KB")
        print(f"Maximum contig size: {self.max_contig_size_bytes / 1024} KB")
        print(f"Output directory: {output_dir}")
        print("="*60)
        
        results = {}
        total_input_files = len(fasta_files)
        total_output_files = 0
        
        for i, fasta_file in enumerate(fasta_files, 1):
            print(f"\n[{i}/{total_input_files}]")
            output_files = self.split_file(fasta_file, output_dir)
            results[fasta_file.name] = output_files
            total_output_files += len(output_files)
        
        print("\n" + "="*60)
        print("Splitting complete!")
        print(f"  Input files: {total_input_files}")
        print(f"  Output files: {total_output_files}")
        print(f"  Tracking files saved alongside output files")
        print("="*60)
        
        return results


# ============================================================================
# PART 2: RESULTS MERGER WITH POSITION CORRECTION
# ============================================================================

class ResultsMerger:
    def __init__(self, split_dir, results_dir):
        """
        Initialize the results merger
        
        Args:
            split_dir: Directory containing split FASTA files and tracking files
            results_dir: Directory containing CRPBSFinder results
        """
        self.split_dir = Path(split_dir)
        self.results_dir = Path(results_dir)
        self.tracking_data = self._load_all_tracking()
    
    def _load_all_tracking(self):
        """Load all tracking files"""
        tracking = {}
        
        tracking_files = list(self.split_dir.glob("*.tracking"))
        print(f"Loading {len(tracking_files)} tracking file(s)...")
        
        for tracking_file in tracking_files:
            with open(tracking_file, 'r') as f:
                reader = csv.DictReader(f, delimiter='\t')
                for row in reader:
                    chunk_name = row['chunk_name']
                    original_contig = row['original_contig']
                    offset = int(row['offset'])
                    tracking[chunk_name] = {
                        'original_contig': original_contig,
                        'offset': offset
                    }
        
        print(f"  Loaded tracking for {len(tracking)} sequence(s)")
        return tracking
    
    def merge_result_file(self, result_file, output_file):
        """
        Merge a single result file, correcting positions
        
        Args:
            result_file: Path to CRPBSFinder result file
            output_file: Path for merged output
        """
        corrected_rows = []
        
        with open(result_file, 'r') as f:
            reader = csv.DictReader(f, delimiter='\t')
            
            for row in reader:
                contig_id = row['ID']
                position = int(row['pos'])
                
                # Check if this is a chunk that needs correction
                if contig_id in self.tracking_data:
                    tracking = self.tracking_data[contig_id]
                    original_contig = tracking['original_contig']
                    offset = tracking['offset']
                    
                    # Correct the position
                    corrected_position = position + offset
                    
                    # Update row
                    row['ID'] = original_contig
                    row['pos'] = str(corrected_position)
                
                corrected_rows.append(row)
        
        # Write corrected results
        if corrected_rows:
            fieldnames = corrected_rows[0].keys()
            with open(output_file, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='\t')
                writer.writeheader()
                writer.writerows(corrected_rows)
            
            return len(corrected_rows)
        return 0
    
    def merge_all_results(self, output_dir=None):
        """
        Merge all result files in the results directory
        
        Args:
            output_dir: Directory for merged results (defaults to results_dir/merged)
        """
        if output_dir is None:
            output_dir = self.results_dir / "merged"
        else:
            output_dir = Path(output_dir)
        
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Find all result files (adjust pattern based on actual output format)
        result_patterns = ['*.txt', '*.tsv', '*.csv']
        result_files = []
        for pattern in result_patterns:
            result_files.extend(self.results_dir.glob(pattern))
        
        # Filter out already merged files and HTML files
        result_files = [f for f in result_files 
                       if 'merged' not in f.name.lower() 
                       and 'combined' not in f.name.lower()
                       and not f.name.endswith('.html')
                       and f.parent.name != 'merged'
                       and f.parent.name != 'combined']
        
        if not result_files:
            print("No result files found!")
            print(f"Looked in: {self.results_dir}")
            return
        
        print("="*60)
        print(f"Found {len(result_files)} result file(s) to merge")
        print(f"Output directory: {output_dir}")
        print("="*60)
        
        total_rows = 0
        for i, result_file in enumerate(result_files, 1):
            print(f"\n[{i}/{len(result_files)}] Processing: {result_file.name}")
            
            output_file = output_dir / f"merged_{result_file.name}"
            
            try:
                num_rows = self.merge_result_file(result_file, output_file)
                total_rows += num_rows
                print(f"  ✓ Corrected {num_rows} row(s), saved to: {output_file.name}")
            except Exception as e:
                print(f"  ✗ Error: {str(e)}")
                import traceback
                traceback.print_exc()
        
        print("\n" + "="*60)
        print(f"Merging complete! Processed {total_rows} total row(s)")
        print(f"Merged files saved to: {output_dir}")
        print("="*60)

    def combine_by_original_file(self, output_dir=None):
        """
        Combine all results that came from the same original FASTA file

        This is useful when one original file was split into multiple parts.
        Files like "sample_part001.fasta", "sample_part002.fasta" will be
        combined into a single "combined_sample.txt" file.

        Args:
            output_dir: Directory for combined results (defaults to same as merged results)
        """
        if output_dir is None:
            # If output_dir is a path like "merged_results", use it directly
            # Otherwise use results_dir/combined
            if isinstance(self.results_dir, Path) and self.results_dir.name == "merged_results":
                output_dir = self.results_dir / "combined"
            else:
                output_dir = Path("merged_results") / "combined"
        else:
            output_dir = Path(output_dir)

        output_dir.mkdir(parents=True, exist_ok=True)

        # Look for merged files in multiple possible locations
        possible_merged_dirs = [
            Path("merged_results"),
            self.results_dir / "merged",
            self.results_dir,
        ]

        merged_files = []
        merged_dir_used = None

        for merged_dir in possible_merged_dirs:
            if merged_dir.exists():
                # Look for merged files
                temp_files = []
                for pattern in ['*.txt', '*.tsv', '*.csv']:
                    temp_files.extend(merged_dir.glob(f"merged_{pattern}"))

                if temp_files:
                    merged_files = temp_files
                    merged_dir_used = merged_dir
                    break

        if not merged_files:
            print("No merged result files found!")
            print("Searched in:")
            for d in possible_merged_dirs:
                print(f"  - {d}")
            return

        print(f"Found merged files in: {merged_dir_used}")

        # Group result files by their base name (before _part###)
        import re
        result_groups = {}

        for result_file in merged_files:
            # Remove "merged_" prefix and file extension
            name = result_file.stem.replace('merged_', '')

            # Remove the random number prefix (e.g., "1773791747")
            # Pattern: starts with digits followed by the actual name
            name = re.sub(r'^\d+', '', name)

            # Remove .fasta or .fa from the middle of the name
            name = re.sub(r'\.fasta|\.fa|\.fna|\.faa', '', name)

            # Extract base name (remove _part### and various suffixes)
            base_name = re.sub(r'_part\d+', '', name)
            base_name = re.sub(r'_22mer_encode_merge_Result_[\d.]+', '', base_name)
            base_name = re.sub(r'_results?$', '', base_name)

            # Clean up any leading/trailing underscores
            base_name = base_name.strip('_')

            if base_name not in result_groups:
                result_groups[base_name] = []
            result_groups[base_name].append(result_file)

        print("="*60)
        print(f"Found {len(result_groups)} original file(s) with results")
        print(f"Output directory: {output_dir}")
        print("="*60)

        # Show grouping preview
        print("\nGrouping preview:")
        for base_name, files in sorted(result_groups.items())[:5]:
            print(f"\n  {base_name}: {len(files)} file(s)")
            for f in files[:3]:
                print(f"    - {f.name}")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")
        if len(result_groups) > 5:
            print(f"\n  ... and {len(result_groups) - 5} more groups")

        print("\n" + "="*60)

        for i, (base_name, files) in enumerate(sorted(result_groups.items()), 1):
            print(f"\n[{i}/{len(result_groups)}] Combining: {base_name}")
            print(f"  From {len(files)} file(s)")

            # Read and combine all rows
            all_rows = []
            fieldnames = None

            for file in sorted(files):
                try:
                    with open(file, 'r') as f:
                        reader = csv.DictReader(f, delimiter='\t')
                        if fieldnames is None:
                            fieldnames = reader.fieldnames

                        for row in reader:
                            all_rows.append(row)
                except Exception as e:
                    print(f"  ⚠ Error reading {file.name}: {str(e)}")

            if all_rows and fieldnames:
                # Sort by contig ID and position
                all_rows.sort(key=lambda x: (x['ID'], int(x['pos'])))

                # Write combined file
                output_file = output_dir / f"combined_{base_name}.txt"

                with open(output_file, 'w', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='\t')
                    writer.writeheader()
                    writer.writerows(all_rows)

                print(f"  ✓ Combined {len(all_rows)} row(s) -> {output_file.name}")
            else:
                print(f"  ⚠ No data to combine")

        print("\n" + "="*60)
        print(f"Combining complete!")
        print(f"Combined files saved to: {output_dir}")
        print("="*60)


# ============================================================================
# PART 3: SINGLE-THREADED AUTOMATION (Original)
# ============================================================================

class CRPBSFinderAutomation:
    def __init__(self, headless=False, download_dir=None):
        """
        Initialize the browser
        
        Args:
            headless: Run browser in headless mode (no GUI)
            download_dir: Directory where files will be downloaded
        """
        self.download_dir = download_dir or os.path.join(os.getcwd(), "results")
        os.makedirs(self.download_dir, exist_ok=True)
        
        # Configure Firefox options
        firefox_options = webdriver.FirefoxOptions()
        
        if headless:
            firefox_options.add_argument('--headless')
        
        # Set download directory
        firefox_options.set_preference("browser.download.folderList", 2)
        firefox_options.set_preference("browser.download.dir", self.download_dir)
        firefox_options.set_preference("browser.download.useDownloadDir", True)
        firefox_options.set_preference("browser.helperApps.neverAsk.saveToDisk", 
                                      "application/octet-stream,text/plain,text/csv,application/zip,application/x-zip-compressed")
        firefox_options.set_preference("browser.download.manager.showWhenStarting", False)
        firefox_options.set_preference("pdfjs.disabled", True)
        
        # Initialize the driver
        self.driver = webdriver.Firefox(
            service=Service(GeckoDriverManager().install()),
            options=firefox_options
        )
        self.wait = WebDriverWait(self.driver, 20)
        self.url = "https://awi.cuhk.edu.cn/~CRPBSFinder/php/index.php"
    
    def upload_file(self, file_path):
        """
        Upload a single FASTA file
        
        Args:
            file_path: Path to the FASTA file
            
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            print(f"\nProcessing: {os.path.basename(file_path)}")
            
            # Navigate to the page
            self.driver.get(self.url)
            
            # Wait for the file input to be present
            file_input = self.wait.until(
                EC.presence_of_element_located((By.ID, "fileToUpload"))
            )
            
            # Upload the file
            absolute_path = os.path.abspath(file_path)
            file_input.send_keys(absolute_path)
            print(f"  ✓ File selected: {os.path.basename(file_path)}")
            
            # Wait a moment for the file to be processed
            time.sleep(1)
            
            # Find and click the submit button
            submit_button = self.wait.until(
                EC.element_to_be_clickable((By.ID, "filesubmit"))
            )
            
            # Get list of files before submission
            files_before = set(os.listdir(self.download_dir))
            
            # Click submit
            submit_button.click()
            print("  ✓ Submitted")
            
            # Wait for results/download to complete
            print("  Waiting for processing and download...")
            
            max_wait = 300
            check_interval = 5
            waited = 0
            
            new_files = []
            while waited < max_wait:
                time.sleep(check_interval)
                waited += check_interval
                
                # Check for new files
                files_after = set(os.listdir(self.download_dir))
                new_files = files_after - files_before
                
                # Filter out temporary files
                new_files = [f for f in new_files if not f.endswith('.part') and not f.endswith('.crdownload')]
                
                if new_files:
                    print(f"  ✓ Downloaded: {', '.join(new_files)}")
                    return True
                
                if waited % 30 == 0:
                    print(f"    Still waiting... ({waited}s)")
            
            # If no files downloaded, save the results page
            print("  ⚠ Processing completed. Results may be displayed on page instead of downloaded.")
            results_file = os.path.join(self.download_dir, f"{Path(file_path).stem}_results.html")
            with open(results_file, 'w', encoding='utf-8') as f:
                f.write(self.driver.page_source)
            print(f"  ✓ Saved results page to: {os.path.basename(results_file)}")
            
            return True
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)}")
            import traceback
            traceback.print_exc()
            return False
    
    def process_directory(self, input_dir, delay=3):
        """
        Process all FASTA files in a directory
        
        Args:
            input_dir: Directory containing FASTA files
            delay: Seconds to wait between submissions
        """
        # Find all FASTA files
        fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
        fasta_files = []
        for ext in fasta_extensions:
            fasta_files.extend(Path(input_dir).glob(ext))
        
        fasta_files = sorted(fasta_files)
        
        if not fasta_files:
            print(f"No FASTA files found in {input_dir}")
            print(f"Looked for extensions: {', '.join(fasta_extensions)}")
            return
        
        print(f"Found {len(fasta_files)} FASTA file(s) to process")
        print(f"Results will be saved to: {self.download_dir}")
        print("="*60)
        
        successful = 0
        failed = 0
        
        for i, fasta_file in enumerate(fasta_files, 1):
            print(f"\n[{i}/{len(fasta_files)}]")
            
            if self.upload_file(str(fasta_file)):
                successful += 1
            else:
                failed += 1
            
            # Wait between submissions
            if i < len(fasta_files):
                print(f"  Waiting {delay} seconds before next upload...")
                time.sleep(delay)
        
        print("\n" + "="*60)
        print(f"Processing complete!")
        print(f"  Successful: {successful}")
        print(f"  Failed: {failed}")
        print(f"  Results saved to: {self.download_dir}")
        print("="*60)
    
    def close(self):
        """Close the browser"""
        self.driver.quit()


# ============================================================================
# PART 4: PARALLEL AUTOMATION WITH AUTO-RECOVERY
# ============================================================================

class ParallelCRPBSFinderAutomation:
    def __init__(self, num_workers=3, headless=True, download_dir="results"):
        """
        Initialize parallel automation
        
        Args:
            num_workers: Number of parallel browser sessions (recommend 2-4)
            headless: Run browsers in headless mode
            download_dir: Base directory for downloads
        """
        self.num_workers = num_workers
        self.headless = headless
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)
        
        # Get the actual Downloads directory
        self.default_download_dir = Path.home() / "Downloads"
        
        # Create a lock for thread-safe operations
        self.lock = threading.Lock()
        
    def create_worker(self, worker_id):
        """Create a single worker (browser session)"""
        # Create a unique download directory for this worker
        worker_download_dir = self.download_dir / f"worker_{worker_id}"
        worker_download_dir.mkdir(parents=True, exist_ok=True)
        
        firefox_options = webdriver.FirefoxOptions()
        
        if self.headless:
            firefox_options.add_argument('--headless')
        
        # Try to set download directory (may not work on all systems)
        firefox_options.set_preference("browser.download.folderList", 2)
        firefox_options.set_preference("browser.download.dir", str(worker_download_dir))
        firefox_options.set_preference("browser.download.useDownloadDir", True)
        firefox_options.set_preference("browser.helperApps.neverAsk.saveToDisk", 
                                      "application/octet-stream,text/plain,text/csv,application/zip,application/x-zip-compressed")
        firefox_options.set_preference("browser.download.manager.showWhenStarting", False)
        firefox_options.set_preference("pdfjs.disabled", True)
        
        driver = webdriver.Firefox(
            service=Service(GeckoDriverManager().install()),
            options=firefox_options
        )
        
        return driver, worker_download_dir
    
    def find_and_move_result_file(self, file_pattern, worker_id, timeout=300):
        """
        Look for result files in ~/Downloads and move them to results directory
        
        Args:
            file_pattern: Part of the filename to match (e.g., the input filename)
            worker_id: ID of the worker
            timeout: Maximum time to wait in seconds
            
        Returns:
            Path to moved file or None if not found
        """
        import time
        
        start_time = time.time()
        check_interval = 5
        
        while (time.time() - start_time) < timeout:
            # Look in ~/Downloads for matching files
            for file in self.default_download_dir.glob("*"):
                # Skip incomplete downloads
                if file.suffix in ['.part', '.crdownload', '.tmp']:
                    continue
                
                # Check if this file matches our pattern and is a result file
                if (file_pattern in file.name and 
                    'Result' in file.name and 
                    file.suffix == '.txt'):
                    
                    # Check if file is still being written (size changing)
                    try:
                        initial_size = file.stat().st_size
                        time.sleep(1)
                        final_size = file.stat().st_size
                        
                        if initial_size != final_size:
                            # File still being written
                            continue
                        
                        # File is complete, move it
                        dest = self.download_dir / file.name
                        
                        # If destination exists, add a number
                        if dest.exists():
                            counter = 1
                            stem = dest.stem
                            suffix = dest.suffix
                            while dest.exists():
                                dest = self.download_dir / f"{stem}_{counter}{suffix}"
                                counter += 1
                        
                        # Move the file
                        file.rename(dest)
                        
                        with self.lock:
                            print(f"[Worker {worker_id}]   ✓ Found and moved: {file.name}")
                        
                        return dest
                    
                    except Exception as e:
                        # File might be locked or still downloading
                        continue
            
            # Wait before checking again
            time.sleep(check_interval)
            
            # Log progress every 30 seconds
            elapsed = int(time.time() - start_time)
            if elapsed % 30 == 0 and elapsed > 0:
                with self.lock:
                    print(f"[Worker {worker_id}]   Still waiting for download... ({elapsed}s)")
        
        return None
    
    def upload_single_file(self, driver, file_path, worker_download_dir, worker_id, max_retries=3):
        """Upload a single file using the provided driver with retry logic"""
        url = "https://awi.cuhk.edu.cn/~CRPBSFinder/php/index.php"
        
        for attempt in range(max_retries):
            try:
                with self.lock:
                    if attempt > 0:
                        print(f"[Worker {worker_id}] Retry {attempt}/{max_retries}: {os.path.basename(file_path)}")
                    else:
                        print(f"[Worker {worker_id}] Processing: {os.path.basename(file_path)}")
                
                # Navigate to the page
                driver.get(url)
                
                # Wait for the file input
                wait = WebDriverWait(driver, 20)
                file_input = wait.until(
                    EC.presence_of_element_located((By.ID, "fileToUpload"))
                )
                
                # Upload the file
                absolute_path = os.path.abspath(file_path)
                file_input.send_keys(absolute_path)
                
                time.sleep(1)
                
                # Find and click submit button
                submit_button = wait.until(
                    EC.element_to_be_clickable((By.ID, "filesubmit"))
                )
                
                submit_button.click()
                
                with self.lock:
                    print(f"[Worker {worker_id}]   ✓ Submitted, waiting for download...")
                
                # Look for the result file using the input filename as pattern
                file_pattern = Path(file_path).stem
                result_file = self.find_and_move_result_file(file_pattern, worker_id, timeout=300)
                
                if result_file:
                    return True
                else:
                    with self.lock:
                        print(f"[Worker {worker_id}]   ⚠ Timeout - no result file found")
                    
                    # Save HTML page as fallback
                    results_file = self.download_dir / f"{Path(file_path).stem}_results.html"
                    with open(results_file, 'w', encoding='utf-8') as f:
                        f.write(driver.page_source)
                    
                    with self.lock:
                        print(f"[Worker {worker_id}]   Saved HTML fallback: {results_file.name}")
                    
                    return False
            
            except Exception as e:
                error_msg = str(e)
                
                # Check if it's a connection error
                if "Connection refused" in error_msg or "HTTPConnectionPool" in error_msg:
                    with self.lock:
                        print(f"[Worker {worker_id}]   ⚠ Browser connection lost")
                    
                    # If we have retries left, wait and try again
                    if attempt < max_retries - 1:
                        with self.lock:
                            print(f"[Worker {worker_id}]   Waiting 10s before retry...")
                        time.sleep(10)
                        continue
                    else:
                        with self.lock:
                            print(f"[Worker {worker_id}]   ✗ Max retries exceeded, skipping file")
                        return False
                else:
                    # Different error, log and fail
                    with self.lock:
                        print(f"[Worker {worker_id}]   ✗ Error: {error_msg}")
                    return False
        
        return False
    
    def worker_function(self, worker_id, file_queue):
        """Function run by each worker thread with browser restart capability"""
        driver = None
        worker_download_dir = None
        
        results = []
        
        for file_idx, file_path in enumerate(file_queue):
            try:
                # Create/recreate driver if needed
                if driver is None:
                    with self.lock:
                        print(f"[Worker {worker_id}] Starting browser...")
                    driver, worker_download_dir = self.create_worker(worker_id)
                
                success = self.upload_single_file(driver, file_path, worker_download_dir, worker_id)
                results.append((file_path, success))
                
                # Small delay between files
                time.sleep(2)
                
            except Exception as e:
                error_msg = str(e)
                
                # If browser connection failed, try to restart it
                if "Connection refused" in error_msg or "HTTPConnectionPool" in error_msg:
                    with self.lock:
                        print(f"[Worker {worker_id}]   Browser crashed, restarting...")
                    
                    try:
                        if driver:
                            driver.quit()
                    except:
                        pass
                    
                    driver = None  # Will be recreated on next iteration
                    results.append((file_path, False))
                    time.sleep(5)
                else:
                    with self.lock:
                        print(f"[Worker {worker_id}]   ✗ Unexpected error: {error_msg}")
                    results.append((file_path, False))
        
        # Cleanup
        try:
            if driver:
                driver.quit()
        except:
            pass
        
        # Clean up worker directory if empty
        try:
            if worker_download_dir and not any(worker_download_dir.iterdir()):
                worker_download_dir.rmdir()
        except:
            pass
        
        return results
    
    def process_directory_parallel(self, input_dir):
        """
        Process all FASTA files in parallel
        
        Args:
            input_dir: Directory containing FASTA files
        """
        # Find all FASTA files
        fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
        fasta_files = []
        for ext in fasta_extensions:
            fasta_files.extend(Path(input_dir).glob(ext))
        
        fasta_files = sorted(fasta_files)
        
        if not fasta_files:
            print(f"No FASTA files found in {input_dir}")
            return
        
        print("="*70)
        print(f"PARALLEL PROCESSING WITH {self.num_workers} WORKERS")
        print("="*70)
        print(f"Total files: {len(fasta_files)}")
        print(f"Monitoring downloads in: {self.default_download_dir}")
        print(f"Moving results to: {self.download_dir}")
        print("="*70 + "\n")
        
        # Distribute files among workers
        file_queues = [[] for _ in range(self.num_workers)]
        for i, file_path in enumerate(fasta_files):
            worker_idx = i % self.num_workers
            file_queues[worker_idx].append(file_path)
        
        # Print distribution
        for i, queue in enumerate(file_queues):
            print(f"Worker {i}: {len(queue)} file(s)")
        print()
        
        # Start timing
        start_time = time.time()
        
        # Run workers in parallel
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            futures = []
            for worker_id, file_queue in enumerate(file_queues):
                if file_queue:  # Only start worker if it has files
                    future = executor.submit(self.worker_function, worker_id, file_queue)
                    futures.append(future)
            
            # Wait for all workers to complete
            all_results = []
            for future in as_completed(futures):
                try:
                    results = future.result()
                    all_results.extend(results)
                except Exception as e:
                    print(f"Worker failed with error: {str(e)}")
        
        # Calculate statistics
        elapsed_time = time.time() - start_time
        successful = sum(1 for _, success in all_results if success)
        failed = len(all_results) - successful
        
        print("\n" + "="*70)
        print("PARALLEL PROCESSING COMPLETE")
        print("="*70)
        print(f"Total files: {len(all_results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {failed}")
        print(f"Time elapsed: {elapsed_time / 60:.1f} minutes")
        if len(all_results) > 0:
            print(f"Average time per file: {elapsed_time / len(all_results):.1f} seconds")
        print("="*70)


# ============================================================================
# PART 5: UTILITY FUNCTIONS
# ============================================================================

def move_existing_downloads_to_results(results_dir="results"):
    """
    Move any existing result files from ~/Downloads to results directory
    Run this before starting parallel processing to clean up
    """
    downloads_dir = Path.home() / "Downloads"
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    print("="*70)
    print("MOVING EXISTING DOWNLOADS")
    print("="*70)
    print(f"Source: {downloads_dir}")
    print(f"Destination: {results_path}")
    print("="*70 + "\n")
    
    moved_count = 0
    
    # Look for result files in Downloads
    for file in downloads_dir.glob("*"):
        # Check if it's a result file
        if ('Result' in file.name and 
            file.suffix == '.txt' and
            not file.name.startswith('.')):
            
            dest = results_path / file.name
            
            # If destination exists, add a number
            if dest.exists():
                counter = 1
                stem = dest.stem
                suffix = dest.suffix
                while dest.exists():
                    dest = results_path / f"{stem}_{counter}{suffix}"
                    counter += 1
            
            try:
                file.rename(dest)
                print(f"✓ Moved: {file.name}")
                moved_count += 1
            except Exception as e:
                print(f"✗ Failed to move {file.name}: {str(e)}")
    
    print(f"\n{'='*70}")
    print(f"Moved {moved_count} file(s)")
    print("="*70 + "\n")
    
    return moved_count


def identify_missing_results(split_dir="split_fasta_files", results_dir="results"):
    """
    Identify which split files are missing result files
    
    This is useful for diagnosing issues or seeing what still needs to be processed
    
    Args:
        split_dir: Directory containing split FASTA files
        results_dir: Directory containing results
        
    Returns:
        List of file paths that are missing results
    """
    split_path = Path(split_dir)
    results_path = Path(results_dir)
    
    if not split_path.exists():
        print(f"✗ Split directory not found: {split_dir}")
        return []
    
    # Find all split FASTA files
    fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
    split_files = []
    for ext in fasta_extensions:
        split_files.extend(split_path.glob(ext))
    
    split_files = sorted(split_files)
    
    # Find all result files
    processed = set()
    if results_path.exists():
        result_patterns = ['*.txt', '*.tsv', '*.csv']
        for pattern in result_patterns:
            for result_file in results_path.glob(pattern):
                if (result_file.is_file() and 
                    not result_file.name.endswith('.html') and
                    'merged' not in result_file.name.lower() and
                    'combined' not in result_file.name.lower()):
                    
                    # Check if any split file name appears in this result filename
                    for split_file in split_files:
                        split_name = split_file.stem
                        if split_name in result_file.name:
                            processed.add(split_name)
                            break
    
    # Find missing results
    missing = []
    for split_file in split_files:
        if split_file.stem not in processed:
            missing.append(split_file)
    
    # Print report
    print("\n" + "="*70)
    print(" "*20 + "MISSING RESULTS REPORT")
    print("="*70)
    print(f"Total split files: {len(split_files)}")
    print(f"Files with results: {len(split_files) - len(missing)}")
    print(f"Files missing results: {len(missing)}")
    print("="*70)
    
    if missing:
        print("\nMissing result files (first 20):")
        for i, f in enumerate(missing[:20], 1):
            print(f"  {i}. {f.name}")
        if len(missing) > 20:
            print(f"  ... and {len(missing) - 20} more")
        
        # Save to file
        missing_file = Path("missing_results.txt")
        with open(missing_file, 'w') as f:
            f.write("Files missing results:\n")
            for file in missing:
                f.write(f"{file.name}\n")
        
        print(f"\n✓ List saved to: {missing_file}")
    else:
        print("\n✓ All files have results!")
    
    print("="*70 + "\n")
    
    return missing

def verify_matching(split_dir="split_fasta_files", results_dir="results"):
    """
    Helper function to verify that result files are being matched correctly
    Shows which split files have matching results
    """
    import re
    
    split_path = Path(split_dir)
    results_path = Path(results_dir)
    
    # Get all split files
    fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
    split_files = []
    for ext in fasta_extensions:
        split_files.extend(split_path.glob(ext))
    
    split_files = sorted(split_files)
    
    # Get all result files
    result_files = []
    for pattern in ['*.txt', '*.tsv', '*.csv']:
        result_files.extend(results_path.glob(pattern))
    
    result_files = [f for f in result_files 
                   if not f.name.endswith('.html')
                   and 'merged' not in f.name.lower()
                   and 'combined' not in f.name.lower()]
    
    print("="*70)
    print("MATCHING VERIFICATION")
    print("="*70)
    print(f"Split files: {len(split_files)}")
    print(f"Result files: {len(result_files)}")
    print("="*70)
    
    # Create a mapping
    matched = {}
    unmatched_results = []
    
    for result_file in result_files:
        found_match = False
        for split_file in split_files:
            if split_file.stem in result_file.name:
                matched[split_file.stem] = result_file.name
                found_match = True
                break
        
        if not found_match:
            unmatched_results.append(result_file.name)
    
    # Find unmatched split files
    unmatched_splits = [f for f in split_files if f.stem not in matched]
    
    print(f"\nMatched: {len(matched)}")
    print(f"Unmatched split files: {len(unmatched_splits)}")
    print(f"Unmatched result files: {len(unmatched_results)}")
    
    # Show examples
    if matched:
        print(f"\nExample matches (first 10):")
        for i, (split_name, result_name) in enumerate(list(matched.items())[:10]):
            print(f"  {split_name}")
            print(f"    -> {result_name}")
    
    if unmatched_splits:
        print(f"\nUnmatched split files (first 10):")
        for f in unmatched_splits[:10]:
            print(f"  ✗ {f.name}")
    
    if unmatched_results:
        print(f"\nUnmatched result files (first 10):")
        for r in unmatched_results[:10]:
            print(f"  ? {r}")
    
    print("="*70)
    
    return matched, unmatched_splits, unmatched_results


def check_workflow_status(input_dir, split_dir="split_fasta_files", 
                         results_dir="results", merged_dir="merged_results"):
    """
    Check the status of a workflow run
    
    Useful for checking progress or diagnosing issues
    """
    print("\n" + "="*70)
    print(" "*25 + "WORKFLOW STATUS")
    print("="*70)
    
    # Check input files
    input_path = Path(input_dir)
    fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
    input_files = []
    for ext in fasta_extensions:
        input_files.extend(input_path.glob(ext))
    
    print(f"\n1. INPUT FILES ({input_dir}):")
    print(f"   Found: {len(input_files)} FASTA file(s)")
    
    # Check split files
    split_path = Path(split_dir)
    if split_path.exists():
        split_files = []
        for ext in fasta_extensions:
            split_files.extend(split_path.glob(ext))
        tracking_files = list(split_path.glob("*.tracking"))
        
        print(f"\n2. SPLIT FILES ({split_dir}):")
        print(f"   FASTA files: {len(split_files)}")
        print(f"   Tracking files: {len(tracking_files)}")
    else:
        print(f"\n2. SPLIT FILES ({split_dir}):")
        print(f"   Directory not found - splitting not yet done")
    
    # Check results
    results_path = Path(results_dir)
    if results_path.exists():
        result_patterns = ['*.txt', '*.tsv', '*.csv']
        result_files = []
        for pattern in result_patterns:
            result_files.extend(results_path.glob(pattern))
        result_files = [f for f in result_files if not f.name.endswith('.html')]
        
        html_files = list(results_path.glob("*.html"))
        
        print(f"\n3. RAW RESULTS ({results_dir}):")
        print(f"   Result files: {len(result_files)}")
        print(f"   HTML files: {len(html_files)}")
    else:
        print(f"\n3. RAW RESULTS ({results_dir}):")
        print(f"   Directory not found - no results yet")
    
    # Check merged results
    merged_path = Path(merged_dir)
    if merged_path.exists():
        merged_files = list(merged_path.glob("merged_*.txt"))
        combined_files = list(merged_path.glob("combined_*.txt"))
        
        print(f"\n4. MERGED RESULTS ({merged_dir}):")
        print(f"   Position-corrected files: {len(merged_files)}")
        print(f"   Combined files: {len(combined_files)}")
        
        if combined_files:
            print(f"\n   Combined result files:")
            for cf in combined_files:
                size = cf.stat().st_size
                print(f"     - {cf.name} ({size} bytes)")
    else:
        print(f"\n4. MERGED RESULTS ({merged_dir}):")
        print(f"   Directory not found - merging not yet done")
    
    print("\n" + "="*70 + "\n")


def check_progress(split_dir="split_fasta_files", results_dir="results"):
    """
    Quick progress check - run this in a separate Python session
    to monitor progress while parallel processing is running
    """
    import time
    
    while True:
        split_path = Path(split_dir)
        results_path = Path(results_dir)
        
        # Count split files
        fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
        split_files = []
        for ext in fasta_extensions:
            split_files.extend(split_path.glob(ext))
        
        # Count result files
        processed = set()
        for pattern in ['*.txt', '*.tsv', '*.csv']:
            for result_file in results_path.glob(pattern):
                if (result_file.is_file() and 
                    not result_file.name.endswith('.html') and
                    'merged' not in result_file.name.lower() and
                    'combined' not in result_file.name.lower()):
                    
                    for split_file in split_files:
                        if split_file.stem in result_file.name:
                            processed.add(split_file.stem)
                            break
        
        total = len(split_files)
        done = len(processed)
        remaining = total - done
        percent = (done / total * 100) if total > 0 else 0
        
        print(f"\r[{time.strftime('%H:%M:%S')}] Progress: {done}/{total} ({percent:.1f}%) - Remaining: {remaining}  ", end="", flush=True)
        
        time.sleep(10)  # Update every 10 seconds


# ============================================================================
# PART 6: HIGH-LEVEL WORKFLOW FUNCTIONS
# ============================================================================

def resume_workflow(split_dir="split_fasta_files", results_dir="results", 
                   merged_dir="merged_results", upload_delay=3, headless=False):
    """
    Resume a workflow from where it left off (single-threaded)
    
    This function will:
    1. Check which split files have already been processed
    2. Upload only the remaining files
    3. Merge all results (including newly downloaded ones)
    
    Args:
        split_dir: Directory containing split FASTA files
        results_dir: Directory containing raw results
        merged_dir: Directory for merged results
        upload_delay: Seconds between uploads
        headless: Run browser in headless mode
    """
    print("\n" + "="*70)
    print(" "*22 + "RESUMING WORKFLOW")
    print("="*70)
    
    split_path = Path(split_dir)
    if not split_path.exists():
        print(f"✗ Split directory not found: {split_dir}")
        print("  Cannot resume - please run complete workflow from beginning")
        return
    
    # Find all split FASTA files
    fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
    split_files = []
    for ext in fasta_extensions:
        split_files.extend(split_path.glob(ext))
    
    split_files = sorted(split_files)
    
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    # Find already processed files using improved matching
    processed = set()
    
    if results_path.exists():
        result_patterns = ['*.txt', '*.tsv', '*.csv']
        
        for pattern in result_patterns:
            for result_file in results_path.glob(pattern):
                if result_file.is_file():
                    if result_file.name.endswith('.html'):
                        continue
                    if 'merged' in result_file.name.lower() or 'combined' in result_file.name.lower():
                        continue
                    
                    # Check if any split file name appears in this result filename
                    for split_file in split_files:
                        split_name = split_file.stem
                        if split_name in result_file.name:
                            processed.add(split_name)
                            break
    
    # Identify missing files
    remaining_files = []
    already_processed = []
    
    for split_file in split_files:
        if split_file.stem in processed:
            already_processed.append(split_file)
        else:
            remaining_files.append(split_file)
    
    print(f"\nTotal split files: {len(split_files)}")
    print(f"Already processed: {len(already_processed)}")
    print(f"Remaining to process: {len(remaining_files)}")
    
    # Show examples
    if already_processed:
        print(f"\nExample processed files (first 5):")
        for f in already_processed[:5]:
            print(f"  ✓ {f.name}")
        if len(already_processed) > 5:
            print(f"  ... and {len(already_processed) - 5} more")
    
    if remaining_files:
        print(f"\nExample remaining files (first 5):")
        for f in remaining_files[:5]:
            print(f"  • {f.name}")
        if len(remaining_files) > 5:
            print(f"  ... and {len(remaining_files) - 5} more")
    
    if not remaining_files:
        print("\n✓ All files already processed!")
        print("  Proceeding to merge results...")
        
        merger = ResultsMerger(split_dir=split_dir, results_dir=results_dir)
        merger.merge_all_results(output_dir=merged_dir)
        merger.combine_by_original_file(output_dir=merged_dir)
        
        print("\n✓ Resume complete!")
        return
    
    # Ask for confirmation
    print("\n" + "="*70)
    response = input(f"Proceed with uploading {len(remaining_files)} file(s)? (y/n): ")
    if response.lower() != 'y':
        print("Resume cancelled.")
        return
    
    # Upload remaining files
    print("\n" + "="*70)
    print("UPLOADING REMAINING FILES")
    print("="*70)
    
    automation = CRPBSFinderAutomation(headless=headless, download_dir=results_dir)
    
    successful = 0
    failed = 0
    
    try:
        for i, fasta_file in enumerate(remaining_files, 1):
            print(f"\n[{i}/{len(remaining_files)}]")
            
            if automation.upload_file(str(fasta_file)):
                successful += 1
            else:
                failed += 1
            
            if i < len(remaining_files):
                print(f"  Waiting {upload_delay} seconds...")
                time.sleep(upload_delay)
        
        print("\n" + "="*70)
        print(f"Upload complete!")
        print(f"  Successful: {successful}")
        print(f"  Failed: {failed}")
        print("="*70)
        
    except KeyboardInterrupt:
        print("\n\n⚠ Upload interrupted by user")
        print(f"  Processed: {successful + failed}/{len(remaining_files)}")
        print("  You can run resume_workflow() again to continue")
    finally:
        automation.close()
    
    # Merge results
    print("\n" + "="*70)
    print("MERGING RESULTS")
    print("="*70)
    
    try:
        merger = ResultsMerger(split_dir=split_dir, results_dir=results_dir)
        merger.merge_all_results(output_dir=merged_dir)
        merger.combine_by_original_file(output_dir=merged_dir)
        
        print("\n✓ Workflow resumed and completed!")
    except Exception as e:
        print(f"\n✗ Error during merging: {str(e)}")
        import traceback
        traceback.print_exc()

def find_processed_files(split_files, results_path):
    """
    Robustly match split files to their result files
    
    Args:
        split_files: List of Path objects for split FASTA files
        results_path: Path to results directory
        
    Returns:
        Set of processed file stems
    """
    processed = set()
    
    if not results_path.exists():
        return processed
    
    result_patterns = ['*.txt', '*.tsv', '*.csv']
    
    for pattern in result_patterns:
        for result_file in results_path.glob(pattern):
            if not result_file.is_file():
                continue
            
            # Skip unwanted files
            if result_file.name.endswith('.html'):
                continue
            if 'merged' in result_file.name.lower() or 'combined' in result_file.name.lower():
                continue
            
            result_name = result_file.name
            
            # Try multiple matching strategies
            for split_file in split_files:
                split_stem = split_file.stem  # e.g., "MG1655_part002"
                split_name = split_file.name  # e.g., "MG1655_part002.fasta"
                
                # Strategy 1: Exact stem match
                if split_stem in result_name:
                    processed.add(split_stem)
                    break
                
                # Strategy 2: Full filename match (including .fasta)
                if split_name in result_name:
                    processed.add(split_stem)
                    break
                
                # Strategy 3: Handle case where .fasta is in the result name
                # Result: "1774887088MG1655_part002.fasta_22mer..."
                # This checks if the split filename appears after any prefix
                if f"{split_stem}.fasta" in result_name or f"{split_stem}.fa" in result_name:
                    processed.add(split_stem)
                    break
    
    return processed
        
def parallel_resume_workflow(split_dir="split_fasta_files", results_dir="results",
                            merged_dir="merged_results", num_workers=3, headless=True):
    """
    Resume a workflow with parallel processing
    Uses improved file matching to recognize downloaded results
    
    Args:
        split_dir: Directory containing split files
        results_dir: Directory for results
        merged_dir: Directory for merged results
        num_workers: Number of parallel workers
        headless: Run browsers in headless mode
    """
    print("\n" + "="*70)
    print(" "*17 + "PARALLEL RESUME WORKFLOW")
    print("="*70)
    
    split_path = Path(split_dir)
    if not split_path.exists():
        print(f"✗ Split directory not found: {split_dir}")
        return
    
    # Find all split files
    fasta_extensions = ['*.fasta', '*.fa', '*.fna', '*.faa']
    split_files = []
    for ext in fasta_extensions:
        split_files.extend(split_path.glob(ext))
    
    split_files = sorted(split_files)
    
    # Find already processed files using robust matching
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    
    processed = find_processed_files(split_files, results_path)
    
    print(f"\nFound {len(processed)} processed file(s)")
    
    # Identify missing files
    missing_files = []
    already_processed = []
    
    for split_file in split_files:
        if split_file.stem in processed:
            already_processed.append(split_file)
        else:
            missing_files.append(split_file)
    
    print(f"\nTotal split files: {len(split_files)}")
    print(f"Already processed: {len(already_processed)}")
    print(f"Remaining to process: {len(missing_files)}")
    
    # Show examples
    if already_processed:
        print(f"\nExample processed files (first 5):")
        for f in already_processed[:5]:
            print(f"  ✓ {f.name}")
        if len(already_processed) > 5:
            print(f"  ... and {len(already_processed) - 5} more")
    
    if missing_files:
        print(f"\nExample remaining files (first 5):")
        for f in missing_files[:5]:
            print(f"  • {f.name}")
        if len(missing_files) > 5:
            print(f"  ... and {len(missing_files) - 5} more")
    
    if not missing_files:
        print("\n✓ All files already processed!")
        print("  Proceeding to merge results...")
        
        merger = ResultsMerger(split_dir=split_dir, results_dir=results_dir)
        merger.merge_all_results(output_dir=merged_dir)
        merger.combine_by_original_file(output_dir=merged_dir)
        
        print("\n✓ Resume complete!")
        return
    
    # Ask for confirmation
    print("\n" + "="*70)
    response = input(f"Process {len(missing_files)} file(s) with {num_workers} workers? (y/n): ")
    if response.lower() != 'y':
        print("Resume cancelled.")
        return
    
    # Create a temporary directory with just the missing files
    import tempfile
    import shutil
    
    temp_dir = Path(tempfile.mkdtemp(prefix="resume_parallel_"))
    
    try:
        # Create symlinks to missing files
        print(f"\nPreparing {len(missing_files)} file(s) for parallel processing...")
        for missing_file in missing_files:
            link_path = temp_dir / missing_file.name
            try:
                link_path.symlink_to(missing_file.absolute())
            except:
                # If symlinks don't work, copy the file
                shutil.copy2(missing_file, link_path)
        
        # Process missing files in parallel
        print("\n" + "="*70)
        print(f"UPLOADING {len(missing_files)} FILES WITH {num_workers} WORKERS")
        print("="*70)
        
        parallel_automation = ParallelCRPBSFinderAutomation(
            num_workers=num_workers,
            headless=headless,
            download_dir=results_dir
        )
        
        parallel_automation.process_directory_parallel(temp_dir)
        
        print(f"\n✓ Parallel upload complete")
        
    except KeyboardInterrupt:
        print("\n\n⚠ Processing interrupted by user")
        print("  You can run parallel_resume_workflow() again to continue")
    except Exception as e:
        print(f"\n✗ Error during processing: {str(e)}")
        import traceback
        traceback.print_exc()
    finally:
        # Clean up temporary directory
        try:
            shutil.rmtree(temp_dir)
        except:
            pass
    
    # Merge results
    print("\n" + "="*70)
    print("MERGING RESULTS")
    print("="*70)
    
    try:
        merger = ResultsMerger(split_dir=split_dir, results_dir=results_dir)
        
        print("\n--- Correcting positions ---")
        merger.merge_all_results(output_dir=merged_dir)
        
        print("\n--- Combining results by original file ---")
        merger.combine_by_original_file(output_dir=merged_dir)
        
        print("\n✓ Parallel resume complete!")
    except Exception as e:
        print(f"\n✗ Error during merging: {str(e)}")
        import traceback
        traceback.print_exc()
        


def parallel_complete_workflow(input_dir, num_workers=3, max_size_kb=30, 
                               max_contig_size_kb=25, headless=True, cleanup=False):
    """
    Complete workflow with parallel processing
    
    Args:
        input_dir: Directory containing original FASTA files
        num_workers: Number of parallel browser sessions (recommend 2-4)
        max_size_kb: Maximum file size
        max_contig_size_kb: Maximum contig size before splitting
        headless: Run browsers in headless mode
        cleanup: Delete intermediate files after completion
    """
    split_dir = "split_fasta_files"
    results_dir = "results"
    merged_dir = "merged_results"
    
    print("\n" + "="*70)
    print(" "*15 + "PARALLEL COMPLETE WORKFLOW")
    print("="*70)
    print(f"Input directory: {input_dir}")
    print(f"Parallel workers: {num_workers}")
    print(f"Split files directory: {split_dir}")
    print(f"Results directory: {results_dir}")
    print(f"Merged results directory: {merged_dir}")
    print("="*70)
    
    # =========================================================================
    # STEP 1: Split FASTA files
    # =========================================================================
    print("\n" + "="*70)
    print("STEP 1: SPLITTING FASTA FILES")
    print("="*70)
    
    splitter = AdvancedFastaSplitter(
        max_size_kb=max_size_kb,
        max_contig_size_kb=max_contig_size_kb
    )
    
    split_results = splitter.split_directory(input_dir, split_dir)
    
    if not split_results:
        print("\n✗ No files were split. Aborting workflow.")
        return
    
    total_split_files = sum(len(files) for files in split_results.values())
    print(f"\n✓ Step 1 complete: Created {total_split_files} file(s)")
    
    time.sleep(2)
    
    # =========================================================================
    # STEP 2: Upload files in parallel
    # =========================================================================
    print("\n" + "="*70)
    print("STEP 2: UPLOADING FILES IN PARALLEL")
    print("="*70)
    
    parallel_automation = ParallelCRPBSFinderAutomation(
        num_workers=num_workers,
        headless=headless,
        download_dir=results_dir
    )
    
    parallel_automation.process_directory_parallel(split_dir)
    
    print(f"\n✓ Step 2 complete: All uploads finished")
    
    time.sleep(2)
    
    # =========================================================================
    # STEP 3: Merge results
    # =========================================================================
    print("\n" + "="*70)
    print("STEP 3: MERGING RESULTS")
    print("="*70)
    
    merger = ResultsMerger(
        split_dir=split_dir,
        results_dir=results_dir
    )
    
    print("\n--- Correcting positions ---")
    merger.merge_all_results(output_dir=merged_dir)
    
    print("\n--- Combining results by original file ---")
    merger.combine_by_original_file(output_dir=merged_dir)
    
    print(f"\n✓ Step 3 complete: Results merged and combined")
    
    # =========================================================================
    # STEP 4: Cleanup (optional)
    # =========================================================================
    if cleanup:
        print("\n" + "="*70)
        print("STEP 4: CLEANUP")
        print("="*70)
        
        import shutil
        
        try:
            print(f"Removing split files directory: {split_dir}")
            shutil.rmtree(split_dir)
            print("✓ Split files removed")
            
            # Remove worker directories
            results_path = Path(results_dir)
            for worker_dir in results_path.glob("worker_*"):
                if worker_dir.is_dir():
                    shutil.rmtree(worker_dir)
            print("✓ Worker directories removed")
            
        except Exception as e:
            print(f"⚠ Cleanup warning: {str(e)}")
    
    # =========================================================================
    # FINAL SUMMARY
    # =========================================================================
    print("\n" + "="*70)
    print(" "*20 + "WORKFLOW COMPLETE!")
    print("="*70)
    print(f"✓ Processed with {num_workers} parallel workers")
    print(f"✓ Results saved to: {merged_dir}")
    print("="*70 + "\n")


# ============================================================================
# PART 7: USAGE EXAMPLES AND MAIN
# ============================================================================

# if __name__ == "__main__":
    
#     # ==========================================================================
#     # RECOMMENDED: Resume partially completed run with parallel processing
#     # ==========================================================================
    
#     # Step 1: Move any existing downloads from ~/Downloads to results
#     move_existing_downloads_to_results(results_dir="results")
    
#     # Step 2: Resume with parallel workers
#     parallel_resume_workflow(
#         split_dir="split_fasta_files",
#         results_dir="results",
#         merged_dir="merged_results",
#         num_workers=5,           # Adjust: 2-4 is recommended
#         headless=True            # Run in background
#     )
    
    # ==========================================================================
    # ALTERNATIVE 1: Check status first
    # ==========================================================================
    # check_workflow_status(
    #     input_dir="path/to/original/files",
    #     split_dir="split_fasta_files",
    #     results_dir="results",
    #     merged_dir="merged_results"
    # )
    
    # ==========================================================================
    # ALTERNATIVE 2: Verify matching and identify missing
    # ==========================================================================
    # verify_matching(
    #     split_dir="split_fasta_files",
    #     results_dir="results"
    # )
    # 
    # missing = identify_missing_results(
    #     split_dir="split_fasta_files",
    #     results_dir="results"
    # )
    
    # ==========================================================================
    # ALTERNATIVE 3: Full workflow from scratch
    # ==========================================================================
    # parallel_complete_workflow(
    #     input_dir="path/to/your/fasta/files",
    #     num_workers=3,              # Number of parallel browsers (2-4 recommended)
    #     max_size_kb=30,             # Maximum file size
    #     max_contig_size_kb=25,      # Maximum contig size
    #     headless=True,              # Run browsers in background
    #     cleanup=False               # Keep intermediate files
    # )
    
    # ==========================================================================
    # ALTERNATIVE 4: Single-threaded resume (slower but more stable)
    # ==========================================================================
    # resume_workflow(
    #     split_dir="split_fasta_files",
    #     results_dir="results",
    #     merged_dir="merged_results",
    #     upload_delay=3,
    #     headless=True
    # )
    
    # ==========================================================================
    # MONITORING: Run in separate terminal to watch progress
    # ==========================================================================
    # check_progress(
    #     split_dir="split_fasta_files",
    #     results_dir="results"
    # )
    

    
    
# # Run pipeline on all fasta files in provided directory:
# parallel_complete_workflow(
#         input_dir="./repeat_analysis/fastq_files/",
#         num_workers=3,              # Number of parallel browsers (2-4 recommended)
#         max_size_kb=30,             # Maximum file size
#         max_contig_size_kb=25,      # Maximum contig size
#         headless=True,              # Run browsers in background
#         cleanup=False               # Keep intermediate files
#     )

# Run pipeline on all fasta files in provided directory:
# parallel_complete_workflow(
#         input_dir="./repeat_analysis/fastq_files/",
#         num_workers=3,              # Number of parallel browsers (2-4 recommended)
#         max_size_kb=30,             # Maximum file size
#         max_contig_size_kb=25,      # Maximum contig size
#         headless=True,              # Run browsers in background
#         cleanup=False               # Keep intermediate files
#     )

In [3]:
check_workflow_status("./repeat_analysis/fastq_files/", split_dir="split_fasta_files", 
                         results_dir="results", merged_dir="merged_results")


                         WORKFLOW STATUS

1. INPUT FILES (./repeat_analysis/fastq_files/):
   Found: 1 FASTA file(s)

2. SPLIT FILES (split_fasta_files):
   FASTA files: 194
   Tracking files: 194

3. RAW RESULTS (results):
   Result files: 387
   HTML files: 0

4. MERGED RESULTS (merged_results):
   Position-corrected files: 387
   Combined files: 0




In [ ]:
# Run pipeline on all fasta files in provided directory:
parallel_complete_workflow(
        input_dir="./repeat_analysis/fastq_files/",
        num_workers=3,              # Number of parallel browsers (2-4 recommended)
        max_size_kb=30,             # Maximum file size
        max_contig_size_kb=25,      # Maximum contig size
        headless=True,              # Run browsers in background
        cleanup=False               # Keep intermediate files
    )


               PARALLEL COMPLETE WORKFLOW
Input directory: ./repeat_analysis/fastq_files/
Parallel workers: 3
Split files directory: split_fasta_files
Results directory: results
Merged results directory: merged_results

STEP 1: SPLITTING FASTA FILES
Found 1 FASTA file(s) to process
Maximum file size: 30.0 KB
Maximum contig size: 25.0 KB
Output directory: split_fasta_files

[1/1]

Processing: QOWM01.1.fasta
  Found 79 contig(s)
    Splitting large contig 'QOWM01000001.1' (651.33 KB)
      -> Created 27 chunk(s)
    Splitting large contig 'QOWM01000002.1' (526.32 KB)
      -> Created 22 chunk(s)
    Splitting large contig 'QOWM01000003.1' (484.66 KB)
      -> Created 20 chunk(s)
    Splitting large contig 'QOWM01000004.1' (460.19 KB)
      -> Created 19 chunk(s)
    Splitting large contig 'QOWM01000005.1' (420.58 KB)
      -> Created 17 chunk(s)
    Splitting large contig 'QOWM01000006.1' (205.70 KB)
      -> Created 9 chunk(s)
    Splitting large contig 'QOWM01000007.1' (171.14 KB)
   

In [7]:
parallel_resume_workflow(split_dir="split_fasta_files", results_dir="results",
                            merged_dir="merged_results", num_workers=3, headless=True)


                 PARALLEL RESUME WORKFLOW

Found 194 processed file(s)

Total split files: 194
Already processed: 194
Remaining to process: 0

Example processed files (first 5):
  ✓ QOWM01.1_part001.fasta
  ✓ QOWM01.1_part002.fasta
  ✓ QOWM01.1_part003.fasta
  ✓ QOWM01.1_part004.fasta
  ✓ QOWM01.1_part005.fasta
  ... and 189 more

✓ All files already processed!
  Proceeding to merge results...
Loading 194 tracking file(s)...
  Loaded tracking for 247 sequence(s)
Found 387 result file(s) to merge
Output directory: merged_results

[1/387] Processing: 1788656941QOWM01.1_part130.fasta_22mer_encode_merge_Result_5.5.txt
  ✓ Corrected 60 row(s), saved to: merged_1788656941QOWM01.1_part130.fasta_22mer_encode_merge_Result_5.5.txt

[2/387] Processing: 1788655486QOWM01.1_part064.fasta_22mer_encode_merge_Result_5.5.txt
  ✓ Corrected 63 row(s), saved to: merged_1788655486QOWM01.1_part064.fasta_22mer_encode_merge_Result_5.5.txt

[3/387] Processing: 1788656824QOWM01.1_part125.fasta_22mer_encode_merg

In [ ]:
## Check if anything is running in the background. If so and you need to interrupt/restart, just kill the kernel. 
import threading
for t in threading.enumerate():
    print(t.name, t.is_alive())

MainThread True
IOPub True
Heartbeat True
Thread-3 True
Thread-4 True
Control True
IPythonHistorySavingThread True
Thread-2 True
Thread-5 True
ThreadPoolExecutor-0_0 False
ThreadPoolExecutor-0_1 True
ThreadPoolExecutor-0_2 True
[Worker 1] Processing: EC3_part105.fasta
[Worker 1]   ✗ Error: Message: File not found: /Users/rlporter/Documents/gut_native_ec_motility/ecor_strains/data_analysis/genetic_variation/CRPprediction/split_fasta_files/EC3_part105.fasta
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:199:5
InvalidArgumentError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:401:5
interaction.uploadFiles@chrome://remote/content/marionette/interaction.sys.mjs:579:13

[Worker 1] Processing: EC3_part108.fasta
[Worker 1]   ✗ Error: Message: File not found: /Users/rlporter/Documents/gut_native_ec_motility/ecor_strains/data_analysis/genetic_variation/CRPprediction/split_fasta_files/